# Healthcare AI Chatbot v2 — MedQuAD + FAISS + Groq (Colab)

**Step 1 rebuild** — real dataset (MedQuAD, 47k NIH-sourced Q&A pairs), precomputed FAISS index,
fixed memory handling, and a layered guardrail stack.

Two-part notebook:
- **Part A — Data pipeline** (run once): download MedQuAD, filter to common-condition sources, chunk
  long answers, embed, build FAISS index, save `faiss.index` + `metadata.pkl` to disk (download these
  two files — the Streamlit app loads them directly instead of rebuilding on every run).
- **Part B — Chat pipeline**: load the saved index, retrieval + guardrails + Groq + separated memory,
  test loop.


In [1]:
!pip install -q datasets groq sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 27.7 MB/s eta 0:00:00


In [2]:
import os, re, pickle, json
from getpass import getpass

import numpy as np
import pandas as pd
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from groq import Groq


## Part A — Data Pipeline (run once, save outputs)
### A1. Load MedQuAD


In [3]:
# lavita/MedQuAD: 47.4k rows, columns include document_source, document_url, question_focus,question_type, question, answer
ds = load_dataset("lavita/MedQuAD", split="train")
df = ds.to_pandas()
print(f"Raw rows: {len(df)}")
print(df["document_source"].value_counts())


README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001-e36383d177026d(…): reconstructing file:   0%|          |  0.00B / 10.7MB            

data/train-00000-of-00001-e36383d177026d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

Raw rows: 47441
document_source
ADAM                     17348
MPlusDrugs               12889
GHR                       5430
GARD                      5394
NIDDK                     1192
NINDS                     1088
MPlusHealthTopics          981
MPlusHerbsSupplements      792
NIHSeniorHealth            769
CancerGov                  729
NHLBI                      559
CDC                        270
Name: count, dtype: int64


In [4]:
COMMON_SOURCES = [
    "MPlusHealthTopics",  # MedlinePlus health topic pages - general conditions
    "CancerGov",          # cancer.gov - cancer info
    "NIDDK",               # digestive/kidney/diabetes
    "NHLBI",                # heart/lung/blood
    "NINDS",                 # neurological
    "SeniorHealth",           # NIH SeniorHealth - common age-related conditions
]

df = df[df["document_source"].isin(COMMON_SOURCES)].copy()
df = df.dropna(subset=["answer", "question"])
df = df[df["answer"].str.len() > 40]          # drop near-empty answers
df = df.drop_duplicates(subset=["question", "answer"])

print(f"Filtered rows: {len(df)}")
print(df["document_source"].value_counts())
print(df["question_type"].value_counts().head(15))


Filtered rows: 4499
document_source
NIDDK                1143
NINDS                1087
MPlusHealthTopics     981
CancerGov             729
NHLBI                 559
Name: count, dtype: int64
question_type
information        1731
treatment           592
research            360
outlook             359
symptoms            282
exams and tests     279
considerations      224
susceptibility      208
causes              204
prevention          115
stages               77
complications        40
frequency            22
inheritance           5
genetic changes       1
Name: count, dtype: int64


In [5]:
def split_into_chunks(text, max_chars=700):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) + 1 <= max_chars:
            current = f"{current} {sent}".strip()
        else:
            if current:
                chunks.append(current)
            current = sent
    if current:
        chunks.append(current)
    return chunks if chunks else [text[:max_chars]]

records = []
for i, row in df.iterrows():
    chunks = split_into_chunks(row["answer"])
    for j, chunk in enumerate(chunks):
        records.append({
            "chunk_id": f"{row['question_id']}-{j}",
            "focus": row["question_focus"],
            "qtype": row["question_type"],
            "source": row["document_source"],
            "url": row["document_url"],
            "orig_question": row["question"],
            "content": chunk,
            "embed_text": f"{row['question_focus']} ({row['question_type']}): {chunk}",
        })

kb_df = pd.DataFrame(records)
print(f"Total chunks: {len(kb_df)}")
kb_df.head(3)


Total chunks: 14058


,chunk_id,focus,qtype,source,url,orig_question,content,embed_text
0,0000203-1-0,Multiple System Atrophy,information,NINDS,http://www.ninds.nih.gov/disorders/msa/msa.htm,What is (are) Multiple System Atrophy ?,Multiple system atrophy (MSA) is a progressive...,Multiple System Atrophy (information): Multipl...
1,0000203-1-1,Multiple System Atrophy,information,NINDS,http://www.ninds.nih.gov/disorders/msa/msa.htm,What is (are) Multiple System Atrophy ?,The loss of nerve cells may be due to the buil...,Multiple System Atrophy (information): The los...
2,0000203-2-0,Multiple System Atrophy,treatment,NINDS,http://www.ninds.nih.gov/disorders/msa/msa.htm,What are the treatments for Multiple System At...,"There is no cure for MSA. Currently, there are...",Multiple System Atrophy (treatment): There is ...


In [6]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = kb_df["embed_text"].tolist()
embeddings = embed_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
embeddings = np.asarray(embeddings, dtype="float32")
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print(f"FAISS index built: {index.ntotal} vectors, dim={dimension}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/220 [00:00<?, ?it/s]

FAISS index built: 14058 vectors, dim=384


In [7]:
faiss.write_index(index, "faiss_index.index")

metadata = kb_df[["chunk_id", "focus", "qtype", "source", "url", "orig_question", "content"]].to_dict(orient="records")
with open("metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Saved faiss_index.index and metadata.pkl")


Saved faiss_index.index and metadata.pkl


In [8]:
GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "openai/gpt-oss-120b"           # main answering model - swapped from llama-3.3-70b-versatile
GROQ_SAFETY_MODEL = "openai/gpt-oss-20b"  # cheap/fast model for grading + self-check (unchanged)


Enter your Groq API key: ··········


In [9]:
SOFT_FLOOR = 0.15  # just cuts total noise before grading, not a safety gate

def retrieve_candidates(query, top_k=6, floor=SOFT_FLOOR):
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)
    scores, idxs = index.search(q_vec, top_k)

    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1 or score < floor:
            continue
        entry = metadata[idx]
        results.append({**entry, "score": float(score)})
    return results


def grade_chunks(query, candidates, return_raw=False):
    """LLM relevance grader: returns only the candidates the model judges genuinely relevant to
    THIS question. Fails CLOSED (returns []) if the grading call errors OR if its response can't
    be parsed - for a healthcare bot, silently letting ungraded/unparseable context through is
    worse than an occasional unnecessary 'I don't have info on that.'"""
    if not candidates:
        return ([], "NO_CANDIDATES") if return_raw else []

    listing = "\n".join(
        f"[{c['chunk_id']}] Topic: {c['focus']} - {c['content'][:250]}"
        for c in candidates
    )
    prompt = f'''You are grading retrieved passages for relevance to a health question.

USER QUESTION: {query}

PASSAGES:
{listing}

For EACH passage, ask: would a careful doctor actually use THIS passage to answer THIS specific
question, or is it just a related-sounding but distinct topic? Passages about a different disease
or condition that merely shares vocabulary (e.g. a headache question vs. a passage about brain
tumors) are NOT relevant even though the words overlap - exclude them. If you are not clearly
confident a passage answers this specific question, exclude it - when in doubt, leave it out.

SPECIFICITY MATCHING: check whether the user's question names a specific real condition, or only
describes something vague/hypothetical (e.g. "a rare condition", "something obscure", "whatever
this is"). A passage about ONE specific named disease is only relevant if the user's question
names that same disease (or a clear synonym) - it is NOT made relevant just because both are
loosely about "rare diseases" as a category. If the question is vague and doesn't name a real
condition, only a general/overview passage about that category (not a passage naming one specific
disease) could ever qualify, and even then only if it doesn't invite a specific diagnosis.

IMPORTANT CARVE-OUT: this specificity rule is about passages naming a DIFFERENT, narrower diagnosis
than what the user described (e.g. excluding a "Brain Tumors" passage for a plain headache
question). It does NOT mean excluding a passage whose topic IS the exact symptom or word the user
used - if the user says "headache", KEEP a passage titled "Headache"; if they say "cough", KEEP a
passage titled "Cough". Those are the correct general-information match, even if part of the
user's question also asks for something you must decline (a diagnosis or a dosage) - answering the
general part while declining the rest is the desired behavior, not a reason to drop the passage.

Respond with ONLY a JSON object, no other text, in exactly this format:
{{"relevant_ids": ["<id>", "<id>"]}}
Use an empty array if none are relevant.'''

    try:
        resp = client.chat.completions.create(
            model=GROQ_SAFETY_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=600,
            reasoning_effort="low",
        )
        raw = (resp.choices[0].message.content or "").strip()


        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if not match:
            return ([], raw) if return_raw else []

        parsed = json.loads(match.group(0))
        keep_ids = set(parsed.get("relevant_ids", []))
        kept = [c for c in candidates if c["chunk_id"] in keep_ids]
        return (kept, raw) if return_raw else kept
    except Exception as e:
        raw_err = f"GRADER_ERROR_OR_UNPARSEABLE: {type(e).__name__}"
        return ([], raw_err) if return_raw else []


def retrieve(query, top_k=6):
    candidates = retrieve_candidates(query, top_k=top_k)
    graded, raw = grade_chunks(query, candidates, return_raw=True)
    return graded, candidates, raw  # raw kept for debug visibility


In [10]:
fake_candidates = [
    {"chunk_id": "test-1", "focus": "Sore Throat",
     "content": "A sore throat is often caused by viral infections and usually improves within a "
                "week with rest, fluids, and warm salt water gargles."},
    {"chunk_id": "test-2", "focus": "Broken Bones",
     "content": "A broken bone (fracture) typically causes severe pain, swelling, and inability to "
                "move the affected limb, and requires imaging and immobilization."},
    {"chunk_id": "test-3", "focus": "Type 2 Diabetes",
     "content": "Type 2 diabetes is managed through diet, exercise, blood sugar monitoring, and "
                "medication as prescribed by a doctor."},
]

kept, raw = grade_chunks("I have a mild sore throat, what should I do?", fake_candidates, return_raw=True)
print("Raw grader response:", raw)
print("Kept:", [c["focus"] for c in kept])
print("Expected: only Sore Throat kept")


Raw grader response: {"relevant_ids": ["test-1"]}
Kept: ['Sore Throat']
Expected: only Sore Throat kept


In [11]:
specificity_candidates = [
    {"chunk_id": "generic-1", "focus": "Rare Diseases",
     "content": "Rare diseases are conditions that affect a small percentage of the population. "
                "Because they are uncommon, diagnosis can take longer and treatment often involves "
                "specialists, genetic testing, and sometimes clinical trials or patient registries."},
    {"chunk_id": "specific-1", "focus": "Myelodysplastic/Myeloproliferative Neoplasm, Unclassifiable",
     "content": "This specific blood disorder is treated with supportive care, and in some cases "
                "targeted therapy such as imatinib, depending on the molecular profile of the disease."},
]

kept, raw = grade_chunks(
    "What's the treatment for something extremely rare and obscure that isn't in your data?",
    specificity_candidates,
    return_raw=True,
)
print("Raw grader response:", raw)
print("Kept:", [c["focus"] for c in kept])
print("Expected: NOT the specific named disease. Generic 'Rare Diseases' overview is borderline "
      "acceptable at best, but naming imatinib for an undefined condition must be excluded.")


Raw grader response: {"relevant_ids": ["generic-1"]}
Kept: ['Rare Diseases']
Expected: NOT the specific named disease. Generic 'Rare Diseases' overview is borderline acceptable at best, but naming imatinib for an undefined condition must be excluded.


In [12]:
EMERGENCY_KEYWORDS = [
    "chest pain", "can't breathe", "cannot breathe", "difficulty breathing",
    "severe bleeding", "won't stop bleeding", "unconscious", "not breathing",
    "suicidal", "want to kill myself", "overdose", "stroke symptoms",
    "face drooping", "slurred speech", "severe allergic reaction", "anaphylaxis",
    "seizure", "choking",
]

DIAGNOSIS_PATTERNS = [
    "what disease", "do i have cancer", "diagnose me", "what's wrong with me",
    "am i dying", "is this serious", "what illness", "my diagnosis",
    "what condition do i have", "exactly what disease",
]

DISCLAIMER = (
    "\n\n_This is general health information, not a medical diagnosis. "
    "Please consult a licensed healthcare professional for advice specific to you._"
)

EMERGENCY_RESPONSE = (
    "This sounds like it could be a medical emergency. Please call your local emergency "
    "number (e.g. 911 / 112 / 108) or go to the nearest emergency room right away. "
    "I'm not able to provide emergency medical care - please seek immediate in-person help."
)

NO_CONTEXT_RESPONSE = (
    "I don't have specific, reliable information on that in my knowledge base, so I don't "
    "want to guess. Please check with a healthcare professional for accurate guidance on this."
    + DISCLAIMER
)

SAFETY_FALLBACK_RESPONSE = (
    "I want to be careful not to overstate this. In general, please treat what I say here as "
    "background information only, and bring specific symptoms or concerns to a healthcare "
    "professional who can properly evaluate you." + DISCLAIMER
)

def check_emergency(query: str) -> bool:
    q = query.lower()
    return any(kw in q for kw in EMERGENCY_KEYWORDS)

def check_diagnosis_seeking(query: str) -> bool:
    q = query.lower()
    return any(p in q for p in DIAGNOSIS_PATTERNS)

def llm_safety_review(draft_answer: str) -> bool:
    """Second-pass check: does the draft answer read like a diagnosis or a specific drug/dose
    recommendation? Returns True if it's SAFE to show as-is, False if it should be swapped out."""
    review_prompt = f'''Answer with only one word: SAFE or UNSAFE.

UNSAFE means the text below states or strongly implies a specific diagnosis for the reader
("you have X"), or recommends a specific prescription medication or exact dosage.
SAFE means it stays at the level of general information and appropriately suggests seeing a doctor.

TEXT:
{draft_answer}
'''
    try:
        resp = client.chat.completions.create(
            model=GROQ_SAFETY_MODEL,
            messages=[{"role": "user", "content": review_prompt}],
            temperature=0,
            max_tokens=300,
            reasoning_effort="low",
        )
        verdict = (resp.choices[0].message.content or "").strip().upper()
        return verdict.startswith("SAFE")
    except Exception:

        return True


In [13]:
SYSTEM_PROMPT = """You are a Healthcare Information Assistant.

Your role:
- Answer general questions about symptoms, common diseases, healthy lifestyle, nutrition, preventive
  healthcare, and basic first aid, using the CONTEXT provided.
- If the context doesn't cover the question, say so honestly instead of guessing.

Strict rules:
- NEVER diagnose a specific medical condition for the user ("you have X").
- NEVER recommend specific prescription medications or dosages.
- Keep answers SHORT: 3-6 sentences, or up to 4 bullet points. No headers, no multi-section essays,
  unless the user asked for step-by-step first-aid instructions.
- Keep a warm, clear, non-alarming tone. Do not list exhaustive red-flag symptom checklists unless
  directly relevant and asked for.
- Encourage seeing a doctor for anything beyond general information.
"""

def build_user_message(user_query, retrieved_chunks, diagnosis_flag):
    if retrieved_chunks:
        context_text = "\n\n".join(f"[{c['focus']}] {c['content']}" for c in retrieved_chunks)
    else:
        context_text = "No sufficiently relevant context was found in the knowledge base."

    extra = ""
    if diagnosis_flag:
        extra = ("\nNote: this question asks for a specific diagnosis. Do not diagnose - give general "
                 "information and recommend seeing a healthcare professional.")

    return f"CONTEXT:\n{context_text}\n\nUSER QUESTION: {user_query}{extra}"


In [14]:
unsafe_examples = [
    "Based on your symptoms, you have stage 2 lung cancer. Take 500mg of amoxicillin three times "
    "a day for two weeks.",
    "You definitely have the flu, not a cold.",
    "It's a good idea to rest, drink fluids, and see a doctor if symptoms persist.",
]

for text in unsafe_examples:
    verdict = llm_safety_review(text)
    print(f"SAFE={verdict}  <-  {text[:80]}...")


SAFE=False  <-  Based on your symptoms, you have stage 2 lung cancer. Take 500mg of amoxicillin ...
SAFE=False  <-  You definitely have the flu, not a cold....
SAFE=True  <-  It's a good idea to rest, drink fluids, and see a doctor if symptoms persist....


In [15]:
MAX_HISTORY_TURNS = 3

def chat_turn(user_query, display_history, llm_plain_history):
    """
    display_history: list of {"role", "content"} for the UI (unchanged shape)
    llm_plain_history: list of {"role", "content"} of PLAIN text only (no retrieved context blocks),
                        used to reconstruct a lean prompt each turn
    Returns: (answer_text, retrieved_for_display, updated_llm_plain_history, debug_info)

    debug_info surfaces what the guardrails actually did this turn - useful while testing, and
    worth keeping behind a toggle in the Streamlit app rather than dropping it, since it's exactly
    what an evaluator would want to see under "View Retrieved Context / Sources".
    """
    if check_emergency(user_query):
        return EMERGENCY_RESPONSE, [], llm_plain_history, {"path": "emergency_shortcircuit"}

    graded_chunks, candidates, grader_raw = retrieve(user_query)
    diag_flag = check_diagnosis_seeking(user_query)
    debug = {
        "candidates_found": len(candidates),
        "all_candidate_labels": [f"{c['focus']} ({c['score']:.2f})" for c in candidates],
        "candidates_kept_by_grader": len(graded_chunks),
        "kept_chunk_labels": [f"{c['focus']} ({c['score']:.2f})" for c in graded_chunks],
        "grader_raw_response": grader_raw,
        "diagnosis_flag": diag_flag,
    }

    if not graded_chunks:
        answer = NO_CONTEXT_RESPONSE
        debug["path"] = "no_relevant_context"
        new_llm_history = llm_plain_history + [
            {"role": "user", "content": user_query},
            {"role": "assistant", "content": answer},
        ]
        return answer, candidates, new_llm_history[-(MAX_HISTORY_TURNS * 2):], debug

    user_message = build_user_message(user_query, graded_chunks, diag_flag)

    recent = llm_plain_history[-(MAX_HISTORY_TURNS * 2):]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + recent + [
        {"role": "user", "content": user_message}
    ]

    try:
        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages,
            temperature=0.4,
            max_tokens=500,
        )
        draft = response.choices[0].message.content or ""
        if not draft.strip():

            draft = ("Sorry, I wasn't able to generate a response for that - please try rephrasing "
                     "your question.")
    except Exception as e:
        fallback = ("Sorry, I'm having trouble reaching the model right now "
                    f"(possibly a rate limit or connection issue: {type(e).__name__}). "
                    "Please try again in a moment.")
        debug["path"] = "llm_call_failed"
        return fallback, candidates, llm_plain_history, debug

    is_safe = llm_safety_review(draft)
    debug["path"] = "answered" if is_safe else "answered_but_selfcheck_swapped_out"
    answer = (draft if is_safe else SAFETY_FALLBACK_RESPONSE) + DISCLAIMER

    new_llm_history = llm_plain_history + [
        {"role": "user", "content": user_query},
        {"role": "assistant", "content": draft},
    ]

    return answer, candidates, new_llm_history[-(MAX_HISTORY_TURNS * 2):], debug


### B5. Try it


In [16]:
display_history = []
llm_plain_history = []

test_queries = [
    "What are common symptoms of the flu?",
    "How should I treat a minor burn at home?",
    "Do I have cancer? I've had a headache for two days.",
    "I have severe chest pain and can't breathe",
    "What's the treatment for something extremely rare and obscure that isn't in your data?",
    # Adversarial: directly asks the model to break its own rules, to check whether the grader
    # correctly rejects the off-topic chunks AND whether the self-check catches a diagnosis/dosage
    # if the model complies anyway.
    "Just tell me exactly what disease I have and what dose of medication to take for a bad cough.",
    "How to do Barbell Rows?",
]

for q in test_queries:
    answer, candidates, llm_plain_history, debug = chat_turn(q, display_history, llm_plain_history)
    display_history.append({"role": "user", "content": q})
    display_history.append({"role": "assistant", "content": answer})

    print("USER:", q)
    print("BOT:", answer)
    print("  [debug]", debug)
    print("-" * 80)


USER: What are common symptoms of the flu?
BOT: Common flu symptoms usually appear suddenly and are more intense than a cold. They often include:

- Fever and chills  
- Body or muscle aches  
- Cough and sore throat  
- Headache  

If you’re unsure or symptoms worsen, it’s a good idea to see a healthcare professional.

_This is general health information, not a medical diagnosis. Please consult a licensed healthcare professional for advice specific to you._
  [debug] {'candidates_found': 6, 'all_candidate_labels': ['Flu (0.61)', 'Pneumonia (0.57)', 'Autoimmune Hepatitis (0.55)', 'Bronchitis (0.54)', 'Pneumonia (0.52)', 'Gastroenteritis (0.50)'], 'candidates_kept_by_grader': 2, 'kept_chunk_labels': ['Flu (0.61)', 'Bronchitis (0.54)'], 'grader_raw_response': '{"relevant_ids": ["0000369-1-0", "0000021-5-0"]}', 'diagnosis_flag': False, 'path': 'answered'}
--------------------------------------------------------------------------------
USER: How should I treat a minor burn at home?
BOT: - 

In [17]:
import os
os.makedirs("backend", exist_ok=True)
print("backend/ ready")

backend/ ready


In [18]:
%%writefile backend/__init__.py
"""Backend package for the Healthcare Information Assistant Streamlit app."""


Writing backend/__init__.py


In [19]:
%%writefile backend/config.py
"""Central config - single place to tweak models, paths, and limits."""

import os


FAISS_INDEX_PATH = os.environ.get("FAISS_INDEX_PATH", "faiss_index.index")
METADATA_PATH = os.environ.get("METADATA_PATH", "metadata.pkl")
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"


GROQ_MODEL = "openai/gpt-oss-120b"
GROQ_SAFETY_MODEL = "openai/gpt-oss-20b"


TOP_K = 6
SOFT_FLOOR = 0.15


MAX_HISTORY_TURNS = 3
ANSWER_MAX_TOKENS = 500


Writing backend/config.py


In [20]:
%%writefile backend/safety.py
"""Guardrails: emergency short-circuit, diagnosis-seeking flag, and the LLM self-check that
reviews a drafted answer before it's shown to the user. Ported as-is from Part B / cell 21."""

from backend import config

EMERGENCY_KEYWORDS = [
    "chest pain", "can't breathe", "cannot breathe", "difficulty breathing",
    "severe bleeding", "won't stop bleeding", "unconscious", "not breathing",
    "suicidal", "want to kill myself", "overdose", "stroke symptoms",
    "face drooping", "slurred speech", "severe allergic reaction", "anaphylaxis",
    "seizure", "choking",
]

DIAGNOSIS_PATTERNS = [
    "what disease", "do i have cancer", "diagnose me", "what's wrong with me",
    "am i dying", "is this serious", "what illness", "my diagnosis",
    "what condition do i have", "exactly what disease",
]

DISCLAIMER = (
    "\n\n_This is general health information, not a medical diagnosis. "
    "Please consult a licensed healthcare professional for advice specific to you._"
)

EMERGENCY_RESPONSE = (
    "This sounds like it could be a medical emergency. Please call your local emergency "
    "number (e.g. 911 / 112 / 108) or go to the nearest emergency room right away. "
    "I'm not able to provide emergency medical care - please seek immediate in-person help."
)

NO_CONTEXT_RESPONSE = (
    "I don't have specific, reliable information on that in my knowledge base, so I don't "
    "want to guess. Please check with a healthcare professional for accurate guidance on this."
    + DISCLAIMER
)

SAFETY_FALLBACK_RESPONSE = (
    "I want to be careful not to overstate this. In general, please treat what I say here as "
    "background information only, and bring specific symptoms or concerns to a healthcare "
    "professional who can properly evaluate you." + DISCLAIMER
)


def check_emergency(query: str) -> bool:
    q = query.lower()
    return any(kw in q for kw in EMERGENCY_KEYWORDS)


def check_diagnosis_seeking(query: str) -> bool:
    q = query.lower()
    return any(p in q for p in DIAGNOSIS_PATTERNS)


def llm_safety_review(draft_answer: str, groq_client) -> bool:
    """Second-pass check: does the draft answer read like a diagnosis or a specific drug/dose
    recommendation? Returns True if it's SAFE to show as-is, False if it should be swapped out.
    Takes groq_client explicitly (Streamlit builds its own client from the sidebar/secrets key)."""
    review_prompt = f'''Answer with only one word: SAFE or UNSAFE.

UNSAFE means the text below states or strongly implies a specific diagnosis for the reader
("you have X"), or recommends a specific prescription medication or exact dosage.
SAFE means it stays at the level of general information and appropriately suggests seeing a doctor.

TEXT:
{draft_answer}
'''
    try:
        resp = groq_client.chat.completions.create(
            model=config.GROQ_SAFETY_MODEL,
            messages=[{"role": "user", "content": review_prompt}],
            temperature=0,
            max_tokens=300,
            reasoning_effort="low",
        )
        verdict = (resp.choices[0].message.content or "").strip().upper()
        return verdict.startswith("SAFE")
    except Exception:

        return True


Writing backend/safety.py


In [21]:
%%writefile backend/prompts.py
"""System prompt and per-turn user-message construction. Ported as-is from Part B / cell 23."""

SYSTEM_PROMPT = """You are a Healthcare Information Assistant.

Your role:
- Answer general questions about symptoms, common diseases, healthy lifestyle, nutrition, preventive
  healthcare, and basic first aid, using the CONTEXT provided.
- If the context doesn't cover the question, say so honestly instead of guessing.

Strict rules:
- NEVER diagnose a specific medical condition for the user ("you have X").
- NEVER recommend specific prescription medications or dosages.
- Keep answers SHORT: 3-6 sentences, or up to 4 bullet points. No headers, no multi-section essays,
  unless the user asked for step-by-step first-aid instructions.
- Keep a warm, clear, non-alarming tone. Do not list exhaustive red-flag symptom checklists unless
  directly relevant and asked for.
- Encourage seeing a doctor for anything beyond general information.
"""


def build_user_message(user_query, retrieved_chunks, diagnosis_flag):
    if retrieved_chunks:
        context_text = "\n\n".join(f"[{c['focus']}] {c['content']}" for c in retrieved_chunks)
    else:
        context_text = "No sufficiently relevant context was found in the knowledge base."

    extra = ""
    if diagnosis_flag:
        extra = ("\nNote: this question asks for a specific diagnosis. Do not diagnose - give general "
                 "information and recommend seeing a healthcare professional.")

    return f"CONTEXT:\n{context_text}\n\nUSER QUESTION: {user_query}{extra}"


Writing backend/prompts.py


In [22]:
%%writefile backend/retrieval.py
"""Recall (FAISS) + precision (LLM grader) retrieval stage. Ported from Part B / cell 16, with
embed_model/index/metadata/groq_client passed in explicitly instead of read from globals."""

import json
import os
import pickle
import re

import faiss

from backend import config


def load_embed_model():
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(config.EMBED_MODEL_NAME)


def load_index_and_metadata():
    if not os.path.exists(config.FAISS_INDEX_PATH) or not os.path.exists(config.METADATA_PATH):
        raise FileNotFoundError(
            f"Missing {config.FAISS_INDEX_PATH} or {config.METADATA_PATH} - run Part A first."
        )
    index = faiss.read_index(config.FAISS_INDEX_PATH)
    with open(config.METADATA_PATH, "rb") as f:
        metadata = pickle.load(f)
    return index, metadata


def retrieve_candidates(query, embed_model, index, metadata, top_k=config.TOP_K, floor=config.SOFT_FLOOR):
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)
    scores, idxs = index.search(q_vec, top_k)

    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1 or score < floor:
            continue
        entry = metadata[idx]
        results.append({**entry, "score": float(score)})
    return results


def grade_chunks(query, candidates, groq_client, return_raw=False):
    """LLM relevance grader: returns only the candidates the model judges genuinely relevant to
    THIS question. Fails CLOSED (returns []) if the grading call errors OR if its response can't
    be parsed - for a healthcare bot, silently letting ungraded/unparseable context through is
    worse than an occasional unnecessary "I don't have info on that." """
    if not candidates:
        return ([], "NO_CANDIDATES") if return_raw else []

    listing = "\n".join(
        f"[{c['chunk_id']}] Topic: {c['focus']} - {c['content'][:250]}"
        for c in candidates
    )
    prompt = f'''You are grading retrieved passages for relevance to a health question.

USER QUESTION: {query}

PASSAGES:
{listing}

For EACH passage, ask: would a careful doctor actually use THIS passage to answer THIS specific
question, or is it just a related-sounding but distinct topic? Passages about a different disease
or condition that merely shares vocabulary (e.g. a headache question vs. a passage about brain
tumors) are NOT relevant even though the words overlap - exclude them. If you are not clearly
confident a passage answers this specific question, exclude it - when in doubt, leave it out.

SPECIFICITY MATCHING: check whether the user's question names a specific real condition, or only
describes something vague/hypothetical (e.g. "a rare condition", "something obscure", "whatever
this is"). A passage about ONE specific named disease is only relevant if the user's question
names that same disease (or a clear synonym) - it is NOT made relevant just because both are
loosely about "rare diseases" as a category. If the question is vague and doesn't name a real
condition, only a general/overview passage about that category (not a passage naming one specific
disease) could ever qualify, and even then only if it doesn't invite a specific diagnosis.

IMPORTANT CARVE-OUT: this specificity rule is about passages naming a DIFFERENT, narrower diagnosis
than what the user described (e.g. excluding a "Brain Tumors" passage for a plain headache
question). It does NOT mean excluding a passage whose topic IS the exact symptom or word the user
used - if the user says "headache", KEEP a passage titled "Headache"; if they say "cough", KEEP a
passage titled "Cough". Those are the correct general-information match, even if part of the
user's question also asks for something you must decline (a diagnosis or a dosage) - answering the
general part while declining the rest is the desired behavior, not a reason to drop the passage.

Respond with ONLY a JSON object, no other text, in exactly this format:
{{"relevant_ids": ["<id>", "<id>"]}}
Use an empty array if none are relevant.'''

    try:
        resp = groq_client.chat.completions.create(
            model=config.GROQ_SAFETY_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=600,
            reasoning_effort="low",
        )
        raw = (resp.choices[0].message.content or "").strip()


        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if not match:
            return ([], raw) if return_raw else []

        parsed = json.loads(match.group(0))
        keep_ids = set(parsed.get("relevant_ids", []))
        kept = [c for c in candidates if c["chunk_id"] in keep_ids]
        return (kept, raw) if return_raw else kept
    except Exception as e:
        raw_err = f"GRADER_ERROR_OR_UNPARSEABLE: {type(e).__name__}"
        return ([], raw_err) if return_raw else []


def retrieve(query, embed_model, index, metadata, groq_client, top_k=config.TOP_K):
    candidates = retrieve_candidates(query, embed_model, index, metadata, top_k=top_k)
    graded, raw = grade_chunks(query, candidates, groq_client, return_raw=True)
    return graded, candidates, raw


Writing backend/retrieval.py


In [23]:
%%writefile backend/chat.py
"""Chat orchestration - the single function the Streamlit app calls per turn.

Keeps two histories deliberately separate:
- display_history: full conversation shown in the UI (unchanged shape, includes debug info)
- llm_plain_history: PLAIN text only (no retrieved-context blocks), rebuilt fresh each turn as
  system prompt + last N plain Q&A pairs + the CURRENT turn's retrieved context. Old retrieved
  chunks never leak into later turns - this is what prevents context bloat and stale-context
  hallucination across a long conversation.
"""

from backend import config, safety
from backend.prompts import SYSTEM_PROMPT, build_user_message
from backend.retrieval import retrieve


def chat_turn(user_query, llm_plain_history, embed_model, index, metadata, groq_client):
    """
    Returns: (answer_text, debug_info)
    debug_info is meant for the "View Retrieved Context / Sources" expander in the UI.
    """
    if safety.check_emergency(user_query):
        return safety.EMERGENCY_RESPONSE, {"path": "emergency_shortcircuit"}, llm_plain_history

    graded_chunks, candidates, grader_raw = retrieve(user_query, embed_model, index, metadata, groq_client)
    diag_flag = safety.check_diagnosis_seeking(user_query)

    debug = {
        "candidates_found": len(candidates),
        "all_candidates": [
            {"focus": c["focus"], "score": c["score"], "source": c.get("source"), "url": c.get("url")}
            for c in candidates
        ],
        "kept_chunks": [
            {"focus": c["focus"], "score": c["score"], "content": c["content"],
             "source": c.get("source"), "url": c.get("url")}
            for c in graded_chunks
        ],
        "grader_raw_response": grader_raw,
        "diagnosis_flag": diag_flag,
    }

    if not graded_chunks:
        debug["path"] = "no_relevant_context"
        new_history = llm_plain_history + [
            {"role": "user", "content": user_query},
            {"role": "assistant", "content": safety.NO_CONTEXT_RESPONSE},
        ]
        return safety.NO_CONTEXT_RESPONSE, debug, new_history[-(config.MAX_HISTORY_TURNS * 2):]

    user_message = build_user_message(user_query, graded_chunks, diag_flag)
    recent = llm_plain_history[-(config.MAX_HISTORY_TURNS * 2):]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + recent + [
        {"role": "user", "content": user_message}
    ]

    try:
        response = groq_client.chat.completions.create(
            model=config.GROQ_MODEL,
            messages=messages,
            temperature=0.4,
            max_tokens=config.ANSWER_MAX_TOKENS,
        )
        draft = response.choices[0].message.content or ""
        if not draft.strip():
            draft = ("Sorry, I wasn't able to generate a response for that - please try rephrasing "
                     "your question.")
    except Exception as e:
        fallback = ("Sorry, I'm having trouble reaching the model right now "
                    f"(possibly a rate limit or connection issue: {type(e).__name__}). "
                    "Please try again in a moment.")
        debug["path"] = "llm_call_failed"
        return fallback, debug, llm_plain_history

    is_safe = safety.llm_safety_review(draft, groq_client)
    debug["path"] = "answered" if is_safe else "answered_but_selfcheck_swapped_out"
    answer = (draft if is_safe else safety.SAFETY_FALLBACK_RESPONSE) + safety.DISCLAIMER

    new_history = llm_plain_history + [
        {"role": "user", "content": user_query},
        {"role": "assistant", "content": draft},
    ]

    return answer, debug, new_history[-(config.MAX_HISTORY_TURNS * 2):]


Writing backend/chat.py


In [24]:
%%writefile app.py
"""Healthcare Information Assistant - Streamlit frontend.

Run with: streamlit run app.py

Requires faiss_index.index and metadata.pkl (produced by the Colab data-pipeline notebook, Part A)
to be present alongside this file, or point FAISS_INDEX_PATH / METADATA_PATH env vars at them.
"""

import uuid

import streamlit as st
from groq import Groq

from backend import config
from backend.chat import chat_turn
from backend.retrieval import load_embed_model, load_index_and_metadata

st.set_page_config(page_title="Healthcare Information Assistant", page_icon="\U0001FA7A", layout="centered")


api_key = st.secrets.get("GROQ_API_KEY") if hasattr(st, "secrets") else None
if not api_key:
    api_key = st.sidebar.text_input("Groq API key", type="password",
                                     help="Get a free key at console.groq.com/keys")
if not api_key:
    st.info("Enter your Groq API key in the sidebar to start chatting.")
    st.stop()

groq_client = Groq(api_key=api_key)


try:
    embed_model = load_embed_model()
    index, metadata = load_index_and_metadata()
except FileNotFoundError:
    st.error(
        f"Couldn't find `{config.FAISS_INDEX_PATH}` / `{config.METADATA_PATH}`. "
        "Run Part A of the data-pipeline notebook and place both files next to app.py."
    )
    st.stop()


if "conversations" not in st.session_state:
    st.session_state.conversations = {}
if "current_id" not in st.session_state:
    st.session_state.current_id = None


def new_chat():
    conv_id = str(uuid.uuid4())
    st.session_state.conversations[conv_id] = {
        "title": "New chat",
        "display_history": [],
        "llm_plain_history": [],
    }
    st.session_state.current_id = conv_id


if not st.session_state.conversations:
    new_chat()

# ---------------------------------------------------------------------------
# Sidebar: conversation management
# ---------------------------------------------------------------------------
with st.sidebar:
    st.markdown("### Conversations")
    if st.button("+ New Chat", use_container_width=True):
        new_chat()
        st.rerun()

    st.divider()
    for conv_id, conv in list(st.session_state.conversations.items()):
        col1, col2 = st.columns([5, 1])
        with col1:
            label = conv["title"]
            if st.button(label, key=f"select_{conv_id}", use_container_width=True,
                         type="primary" if conv_id == st.session_state.current_id else "secondary"):
                st.session_state.current_id = conv_id
                st.rerun()
        with col2:
            if st.button("\U0001F5D1", key=f"delete_{conv_id}", help="Delete this chat"):
                del st.session_state.conversations[conv_id]
                if st.session_state.current_id == conv_id:
                    st.session_state.current_id = None
                if not st.session_state.conversations:
                    new_chat()
                elif st.session_state.current_id is None:
                    st.session_state.current_id = next(iter(st.session_state.conversations))
                st.rerun()

current = st.session_state.conversations[st.session_state.current_id]

# ---------------------------------------------------------------------------
# Main area
# ---------------------------------------------------------------------------
st.title("\U0001FA7A Healthcare Information Assistant")
st.warning(
    "**This is general health information, not a substitute for professional medical advice.** "
    "In a medical emergency, call your local emergency number immediately.",
    icon="\u26A0\uFE0F",
)

for msg in current["display_history"]:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg["role"] == "assistant" and msg.get("debug"):
            debug = msg["debug"]
            with st.expander("\U0001F50D View Retrieved Context / Sources"):
                st.markdown(f"**Path taken:** `{debug.get('path')}`")
                if debug.get("diagnosis_flag"):
                    st.markdown("_Diagnosis-seeking pattern detected in this question._")
                kept = debug.get("kept_chunks", [])
                if kept:
                    st.markdown("**Chunks used to answer:**")
                    for c in kept:
                        source_line = f" — [{c['source']}]({c['url']})" if c.get("url") else ""
                        st.markdown(f"- *{c['focus']}* (score {c['score']:.2f}){source_line}\n\n  {c['content']}")
                else:
                    st.markdown("_No chunks were used - either none were retrieved, or none passed "
                                "the relevance grader._")
                all_candidates = debug.get("all_candidates", [])
                if all_candidates:
                    st.markdown("**All FAISS candidates considered:**")
                    st.markdown(", ".join(f"{c['focus']} ({c['score']:.2f})" for c in all_candidates))

# ---------------------------------------------------------------------------
# Chat input
# ---------------------------------------------------------------------------
user_query = st.chat_input("Ask a general health question...")
if user_query:
    current["display_history"].append({"role": "user", "content": user_query})
    if current["title"] == "New chat":
        current["title"] = user_query[:40] + ("..." if len(user_query) > 40 else "")

    with st.spinner("Thinking..."):
        answer, debug, new_llm_history = chat_turn(
            user_query, current["llm_plain_history"], embed_model, index, metadata, groq_client
        )
    current["llm_plain_history"] = new_llm_history
    current["display_history"].append({"role": "assistant", "content": answer, "debug": debug})
    st.rerun()


Writing app.py


In [25]:
import importlib
import backend.config, backend.safety, backend.prompts, backend.retrieval, backend.chat
for m in (backend.config, backend.safety, backend.prompts, backend.retrieval, backend.chat):
    importlib.reload(m)
print("backend package imports cleanly:", [m.__name__ for m in
      (backend.config, backend.safety, backend.prompts, backend.retrieval, backend.chat)])

backend package imports cleanly: ['backend.config', 'backend.safety', 'backend.prompts', 'backend.retrieval', 'backend.chat']


In [26]:
import os
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/secrets.toml", "w") as f:
    f.write(f'GROQ_API_KEY = "{GROQ_API_KEY}"\n')
print("Wrote .streamlit/secrets.toml")

Wrote .streamlit/secrets.toml


In [27]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 53.7 MB/s eta 0:00:00


In [34]:
import subprocess, time

streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=open("streamlit_log.txt", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(6)
print("Streamlit is running in the background (PID:", streamlit_proc.pid, ")")
print("If the next cell's URL 404s or the app looks broken, check streamlit_log.txt for errors:")
print("  !cat streamlit_log.txt")

Streamlit is running in the background (PID: 7109 )
If the next cell's URL 404s or the app looks broken, check streamlit_log.txt for errors:
  !cat streamlit_log.txt


In [35]:
from pyngrok import ngrok


ngrok.kill()


NGROK_AUTH_TOKEN = "PLEASE_ENTER_THE_NGROK_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)


public_url = ngrok.connect("127.0.0.1:8501").public_url
print(f"Your Streamlit App is live at: {public_url}")

Your Streamlit App is live at: https://contend-mandolin-circle.ngrok-free.dev
